# Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch torchvision numpy pillow tqdm
!pip install clean-fid
!pip install scikit-image
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.8 MB/s eta 0:00:00


# Evaluate

## FID, KID

In [ ]:
from cleanfid import fid

path_real = "/content/drive/MyDrive/VTON/inputs"
path_fake = "/content/drive/MyDrive/VTON/outputs"

fid_score = fid.compute_fid(path_real, path_fake)
kid_score = fid.compute_kid(path_real, path_fake)  # ✅ use fid module for both

print(f"FID: {fid_score:.4f}")
print(f"KID: {kid_score:.4f}")


compute FID between two folders


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Found 114 images in the folder /content/drive/MyDrive/VTON/inputs


FID inputs : 100%|██████████| 4/4 [00:13<00:00,  3.48s/it]


Found 114 images in the folder /content/drive/MyDrive/VTON/outputs


...

compute KID between two folders
Found 114 images in the folder /content/drive/MyDrive/VTON/inputs


KID inputs : 100%|██████████| 4/4 [00:07<00:00,  1.86s/it]


Found 114 images in the folder /content/drive/MyDrive/VTON/outputs


KID outputs : 100%|██████████| 4/4 [00:08<00:00,  2.20s/it]


FID: 65.9194
KID: 0.0014


## SSIM

In [ ]:
import os
from skimage.metrics import structural_similarity as ssim
from PIL import Image
import numpy as np
from tqdm import tqdm

path_real = "/content/drive/MyDrive/VTON/inputs"
path_fake = "/content/drive/MyDrive/VTON/outputs"

scores = []
skipped = 0

for i,name in tqdm(enumerate(os.listdir(path_real))):
    real_path = os.path.join(path_real, name)
    output_name = "output" + str(i) + ".jpg"
    fake_path = os.path.join(path_fake, output_name)
    if not os.path.exists(fake_path):
        print(f"❌ Missing: {fake_path}")
        continue

    img1 = np.array(Image.open(real_path).convert("RGB").resize((256, 256)), dtype=np.float32) / 255.0
    img2 = np.array(Image.open(fake_path).convert("RGB").resize((256, 256)), dtype=np.float32) / 255.0

    if img1.std() == 0 or img2.std() == 0:
        print(f"⚠️ Constant image skipped: {name}")
        skipped += 1
        continue

    score = ssim(img1, img2, channel_axis=-1, data_range=1.0)
    if np.isnan(score):
        print(f"⚠️ NaN SSIM for: {name}")
        skipped += 1
        continue

    scores.append(score)

print(f"\n✅ Valid pairs: {len(scores)} | Skipped: {skipped}")
if len(scores) > 0:
    print(f"Average SSIM: {np.mean(scores):.4f}")
else:
    print("❌ No valid image pairs found.")


114it [00:06, 17.12it/s]


✅ Valid pairs: 114 | Skipped: 0
Average SSIM: 0.5233


## RECALL

## Paired

### PostVTON

In [3]:
import torch
import numpy as np

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

In [ ]:
import os


files = []
path_real = f"/content/drive/MyDrive/DressCode/test/{category}"
path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/postvton/final/paired/upper_body/"

for i, name in enumerate(os.listdir(path_real)):
  if "_1" in name:
    continue

  real_file = os.path.join(path_real, name)
  fake_file = os.path.join(path_fake, name)

  if not os.path.isfile(fake_file):
    continue
  else:
    files.append(name)

In [ ]:
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from ultralytics import YOLO
import cv2
import os
import numpy as np

CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.45
IMG_SIZE = 640


def process_one_file(args):
    name, path_real, path_fake = args

    # Load model INSIDE process
    model = YOLO("yolo11l-seg.pt")

    # Get person class ID
    person_class_id = next(
        cls_id for cls_id, cls_name in model.names.items()
        if cls_name.lower() == "person"
    )

    real_path = os.path.join(path_real, name)
    fake_path = os.path.join(path_fake, name)

    img_real = cv2.imread(real_path)
    img_fake = cv2.imread(fake_path)

    if img_real is None or img_fake is None:
        return None

    det_real = model.predict(
        img_real,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cpu"
    )

    det_fake = model.predict(
        img_fake,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cpu"
    )

    boxes_real = det_real[0].boxes.xyxy.cpu().numpy()
    classes_real = det_real[0].boxes.cls.cpu().numpy()
    confs_real = det_real[0].boxes.conf.cpu().numpy()

    boxes_fake = det_fake[0].boxes.xyxy.cpu().numpy()
    classes_fake = det_fake[0].boxes.cls.cpu().numpy()
    confs_fake = det_fake[0].boxes.conf.cpu().numpy()

    non_person_real = [
        (int(cls), float(conf))
        for cls, conf in zip(classes_real, confs_real)
        if int(cls) != person_class_id
    ]

    non_person_fake = [
        (int(cls), float(conf))
        for cls, conf in zip(classes_fake, confs_fake)
        if int(cls) != person_class_id
    ]

    non_person_info = None
    if non_person_real or non_person_fake:
        detected = {}
        for cls, conf in non_person_real:
            detected.setdefault(cls, {"real": [], "fake": []})["real"].append(conf)
        for cls, conf in non_person_fake:
            detected.setdefault(cls, {"real": [], "fake": []})["fake"].append(conf)

        non_person_info = {
            "name": name,
            "classes": detected,
            "real_count": len(non_person_real),
            "fake_count": len(non_person_fake),
        }

    return {
        "name": name,
        "boxes_real": boxes_real,
        "classes_real": classes_real,
        "boxes_fake": boxes_fake,
        "classes_fake": classes_fake,
        "non_person": non_person_info,
    }

path_real = "/content/drive/MyDrive/DressCode/upper_body/images/"
path_fake = "/content/drive/MyDrive/DressCode/postvton_experiment/postvton/final/paired/upper_body/"

args = [(name, path_real, path_fake) for name in files]

results = []
images_with_non_person = []

with Pool(processes=cpu_count()) as pool:
    for out in tqdm(pool.imap_unordered(process_one_file, args), total=len(args)):
        if out is None:
            continue
        results.append((
            out["name"],
            out["boxes_real"],
            out["classes_real"],
            out["boxes_fake"],
            out["classes_fake"]
        ))
        if out["non_person"] is not None:
            images_with_non_person.append(out["non_person"])

100%|██████████| 1800/1800 [2:24:16<00:00,  4.81s/it]


### Lower body

In [ ]:
category = "dresses"

In [ ]:
import os


files = []
path_real = f"/content/drive/MyDrive/DressCode/{category}/images/"
path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/postvton/final/paired/{category}/"

for i, name in enumerate(os.listdir(path_real)):
  if "_1" in name:
    continue

  real_file = os.path.join(path_real, name)
  fake_file = os.path.join(path_fake, name)

  if not os.path.isfile(fake_file):
    continue
  else:
    files.append(name)

In [ ]:
import numpy as np
from ultralytics import YOLO
import cv2
import os
from tqdm import tqdm
import gc

# Load model once to GPU
model = YOLO("yolo11l-seg.pt")
model.to('cuda')

# Parameters
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.45
IMG_SIZE = 640

path_real = f"/content/drive/MyDrive/DressCode/{category}/images/"
path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/postvton/final/unpaired/{category}/"

# Get person class ID
person_class_id = next(
    cls_id for cls_id, cls_name in model.names.items()
    if cls_name.lower() == "person"
)

print(f"Person class ID: {person_class_id}")
print(f"Processing {len(files)} images on GPU...")

results = []
images_with_non_person = []

for name in tqdm(files, desc="Processing"):
    real_path = os.path.join(path_real, name)
    fake_path = os.path.join(path_fake, name)

    img_real = cv2.imread(real_path)
    img_fake = cv2.imread(fake_path)

    if img_real is None or img_fake is None or img_real.size == 0 or img_fake.size == 0:
        print(f"Skipping {name} - invalid image")
        continue

    # GPU inference
    det_real = model.predict(
        img_real,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cuda"
    )

    det_fake = model.predict(
        img_fake,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cuda"
    )

    # Extract detections
    boxes_real = det_real[0].boxes.xyxy.cpu().numpy()
    classes_real = det_real[0].boxes.cls.cpu().numpy()
    confs_real = det_real[0].boxes.conf.cpu().numpy()

    boxes_fake = det_fake[0].boxes.xyxy.cpu().numpy()
    classes_fake = det_fake[0].boxes.cls.cpu().numpy()
    confs_fake = det_fake[0].boxes.conf.cpu().numpy()

    # Check for non-person detections
    non_person_real = [(cls, conf) for cls, conf in zip(classes_real, confs_real)
                       if int(cls) != person_class_id]
    non_person_fake = [(cls, conf) for cls, conf in zip(classes_fake, confs_fake)
                       if int(cls) != person_class_id]

    if len(non_person_real) > 0 or len(non_person_fake) > 0:
        detected_classes = {}
        for cls, conf in non_person_real:
            cls_name = model.names[int(cls)]
            if cls_name not in detected_classes:
                detected_classes[cls_name] = {'real': [], 'fake': []}
            detected_classes[cls_name]['real'].append(f"{conf:.2f}")

        for cls, conf in non_person_fake:
            cls_name = model.names[int(cls)]
            if cls_name not in detected_classes:
                detected_classes[cls_name] = {'real': [], 'fake': []}
            detected_classes[cls_name]['fake'].append(f"{conf:.2f}")

        images_with_non_person.append({
            'name': name,
            'classes': detected_classes,
            'real_count': len(non_person_real),
            'fake_count': len(non_person_fake)
        })

    results.append((name, boxes_real, classes_real, boxes_fake, classes_fake))

# Clear GPU memory
gc.collect()
import torch
torch.cuda.empty_cache()

def iou(box1, box2):
    """Calculate Intersection over Union"""
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    box1Area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2Area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    return interArea / (box1Area + box2Area - interArea + 1e-6)

# Calculate recall
recalls = []
per_class_recalls = {}

for name, boxes_real, classes_real, boxes_fake, classes_fake in results:
    kept = 0

    for r_box, r_cls in zip(boxes_real, classes_real):
        cls_id = int(r_cls)

        if cls_id not in per_class_recalls:
            per_class_recalls[cls_id] = []

        matched = False
        for f_box, f_cls in zip(boxes_fake, classes_fake):
            if r_cls == f_cls and iou(r_box, f_box) > 0.5:
                kept += 1
                per_class_recalls[cls_id].append(1)
                matched = True
                break

        if not matched:
            per_class_recalls[cls_id].append(0)

    total = len(boxes_real)
    if total > 0:
        recalls.append(kept / total)

# Print results
print(f"\n{'='*60}")
print(f"IMAGES WITH NON-PERSON DETECTIONS: {len(images_with_non_person)}")
print(f"Confidence threshold: {CONF_THRESHOLD}")
print(f"{'='*60}")

for img_info in images_with_non_person:
    print(f"\n{img_info['name']}")
    for cls_name, detections in img_info['classes'].items():
        real_confs = ', '.join(detections['real']) if detections['real'] else 'none'
        fake_confs = ', '.join(detections['fake']) if detections['fake'] else 'none'
        print(f"  {cls_name}:")
        print(f"    Real: [{real_confs}] | Fake: [{fake_confs}]")

print(f"\n{'='*60}")
print(f"RECALL METRICS")
print(f"{'='*60}")
if recalls:
    print(f"Overall Accessory Recall: {np.mean(recalls):.4f}")
    print(f"Std Dev: {np.std(recalls):.4f}")
    print(f"Images processed: {len(recalls)}")
else:
    print("No accessories detected in any images")

if per_class_recalls:
    print("\nPer-class recall:")
    for cls_id in sorted(per_class_recalls.keys()):
        if per_class_recalls[cls_id]:
            class_recall = np.mean(per_class_recalls[cls_id])
            class_name = model.names.get(cls_id, f"Class_{cls_id}")
            print(f"  {class_name} (ID={cls_id}): {class_recall:.4f} ({len(per_class_recalls[cls_id])} instances)")

### Unpaired

In [ ]:
category = "dresses"

In [ ]:
import os


files = []
path_real = f"/content/drive/MyDrive/DressCode/{category}/images/"
path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/postvton/final/unpaired/{category}/"

for i, name in enumerate(os.listdir(path_real)):
  if "_1" in name:
    continue

  real_file = os.path.join(path_real, name)
  fake_file = os.path.join(path_fake, name)

  if not os.path.isfile(fake_file):
    continue
  else:
    files.append(name)

In [ ]:
import numpy as np
from ultralytics import YOLO
import cv2
import os
from tqdm import tqdm
import gc

# Load model once to GPU
model = YOLO("yolo11l-seg.pt")
model.to('cuda')

# Parameters
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.45
IMG_SIZE = 640

path_real = f"/content/drive/MyDrive/DressCode/{category}/images/"
path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/postvton/final/unpaired/{category}/"

# Get person class ID
person_class_id = next(
    cls_id for cls_id, cls_name in model.names.items()
    if cls_name.lower() == "person"
)

print(f"Person class ID: {person_class_id}")
print(f"Processing {len(files)} images on GPU...")

results = []
images_with_non_person = []

for name in tqdm(files, desc="Processing"):
    real_path = os.path.join(path_real, name)
    fake_path = os.path.join(path_fake, name)

    img_real = cv2.imread(real_path)
    img_fake = cv2.imread(fake_path)

    if img_real is None or img_fake is None or img_real.size == 0 or img_fake.size == 0:
        print(f"Skipping {name} - invalid image")
        continue

    # GPU inference
    det_real = model.predict(
        img_real,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cuda"
    )

    det_fake = model.predict(
        img_fake,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cuda"
    )

    # Extract detections
    boxes_real = det_real[0].boxes.xyxy.cpu().numpy()
    classes_real = det_real[0].boxes.cls.cpu().numpy()
    confs_real = det_real[0].boxes.conf.cpu().numpy()

    boxes_fake = det_fake[0].boxes.xyxy.cpu().numpy()
    classes_fake = det_fake[0].boxes.cls.cpu().numpy()
    confs_fake = det_fake[0].boxes.conf.cpu().numpy()

    # Check for non-person detections
    non_person_real = [(cls, conf) for cls, conf in zip(classes_real, confs_real)
                       if int(cls) != person_class_id]
    non_person_fake = [(cls, conf) for cls, conf in zip(classes_fake, confs_fake)
                       if int(cls) != person_class_id]

    if len(non_person_real) > 0 or len(non_person_fake) > 0:
        detected_classes = {}
        for cls, conf in non_person_real:
            cls_name = model.names[int(cls)]
            if cls_name not in detected_classes:
                detected_classes[cls_name] = {'real': [], 'fake': []}
            detected_classes[cls_name]['real'].append(f"{conf:.2f}")

        for cls, conf in non_person_fake:
            cls_name = model.names[int(cls)]
            if cls_name not in detected_classes:
                detected_classes[cls_name] = {'real': [], 'fake': []}
            detected_classes[cls_name]['fake'].append(f"{conf:.2f}")

        images_with_non_person.append({
            'name': name,
            'classes': detected_classes,
            'real_count': len(non_person_real),
            'fake_count': len(non_person_fake)
        })

    results.append((name, boxes_real, classes_real, boxes_fake, classes_fake))

# Clear GPU memory
gc.collect()
import torch
torch.cuda.empty_cache()

def iou(box1, box2):
    """Calculate Intersection over Union"""
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    box1Area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2Area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    return interArea / (box1Area + box2Area - interArea + 1e-6)

# Calculate recall
recalls = []
per_class_recalls = {}

for name, boxes_real, classes_real, boxes_fake, classes_fake in results:
    kept = 0

    for r_box, r_cls in zip(boxes_real, classes_real):
        cls_id = int(r_cls)

        if cls_id not in per_class_recalls:
            per_class_recalls[cls_id] = []

        matched = False
        for f_box, f_cls in zip(boxes_fake, classes_fake):
            if r_cls == f_cls and iou(r_box, f_box) > 0.5:
                kept += 1
                per_class_recalls[cls_id].append(1)
                matched = True
                break

        if not matched:
            per_class_recalls[cls_id].append(0)

    total = len(boxes_real)
    if total > 0:
        recalls.append(kept / total)


print(f"\n{'='*60}")
print(f"RECALL METRICS")
print(f"{'='*60}")
if recalls:
    print(f"Overall Accessory Recall: {np.mean(recalls):.4f}")
    print(f"Std Dev: {np.std(recalls):.4f}")
    print(f"Images processed: {len(recalls)}")
else:
    print("No accessories detected in any images")

In [ ]:
import numpy as np
from ultralytics import YOLO
import cv2
import os
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

# Parameters
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.45
IMG_SIZE = 640

# Global variables for worker processes
_model = None
_person_class_id = None


def prepare_images(model, category):
  files = []
  path_real = f"/content/drive/MyDrive/DressCode/{category}/images/"
  path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/{model}/final/unpaired/{category}/"

  for i, name in enumerate(os.listdir(path_real)):
    if "_1" in name:
      continue

    real_file = os.path.join(path_real, name)
    fake_file = os.path.join(path_fake, name)

    if not os.path.isfile(fake_file):
      continue
    else:
      files.append(name)
  return files


def init_worker():
    """Initialize model once per worker process"""
    global _model, _person_class_id
    _model = YOLO("yolo11l-seg.pt")
    _person_class_id = next(
        cls_id for cls_id, cls_name in _model.names.items()
        if cls_name.lower() == "person"
    )

def process_image_pair(args):
    """Process one image pair"""
    name, path_real, path_fake = args

    real_path = os.path.join(path_real, name)
    fake_path = os.path.join(path_fake, name)

    img_real = cv2.imread(real_path)
    img_fake = cv2.imread(fake_path)

    if img_real is None or img_fake is None or img_real.size == 0 or img_fake.size == 0:
        return None

    # Run detection
    det_real = _model.predict(
        img_real,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cpu"
    )

    det_fake = _model.predict(
        img_fake,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cpu"
    )

    # Extract detections
    boxes_real = det_real[0].boxes.xyxy.cpu().numpy()
    classes_real = det_real[0].boxes.cls.cpu().numpy()
    confs_real = det_real[0].boxes.conf.cpu().numpy()

    boxes_fake = det_fake[0].boxes.xyxy.cpu().numpy()
    classes_fake = det_fake[0].boxes.cls.cpu().numpy()
    confs_fake = det_fake[0].boxes.conf.cpu().numpy()

    # Check for non-person detections
    non_person_real = [(int(cls), float(conf)) for cls, conf in zip(classes_real, confs_real)
                       if int(cls) != _person_class_id]
    non_person_fake = [(int(cls), float(conf)) for cls, conf in zip(classes_fake, confs_fake)
                       if int(cls) != _person_class_id]

    non_person_info = None
    if non_person_real or non_person_fake:
        detected_classes = {}
        for cls, conf in non_person_real:
            cls_name = _model.names[int(cls)]
            if cls_name not in detected_classes:
                detected_classes[cls_name] = {'real': [], 'fake': []}
            detected_classes[cls_name]['real'].append(f"{conf:.2f}")

        for cls, conf in non_person_fake:
            cls_name = _model.names[int(cls)]
            if cls_name not in detected_classes:
                detected_classes[cls_name] = {'real': [], 'fake': []}
            detected_classes[cls_name]['fake'].append(f"{conf:.2f}")

        non_person_info = {
            'name': name,
            'classes': detected_classes,
            'real_count': len(non_person_real),
            'fake_count': len(non_person_fake)
        }

    return {
        'name': name,
        'boxes_real': boxes_real,
        'classes_real': classes_real,
        'boxes_fake': boxes_fake,
        'classes_fake': classes_fake,
        'non_person': non_person_info
    }

def iou(box1, box2):
    """Calculate Intersection over Union"""
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    box1Area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2Area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    return interArea / (box1Area + box2Area - interArea + 1e-6)

if __name__ == '__main__':
    model = "OOT"
    categories = ["upper_body", "lower_body", "dressses"]
    for category in categories:
      path_real = f"/content/drive/MyDrive/DressCode/{category}/images/"
      path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/{model}/final/unpaired/{category}/"

      # Prepare arguments
      args = [(name, path_real, path_fake) for name in files]

      num_processes = cpu_count()
      print(f"Processing {len(files)} images using {num_processes} CPU cores...")

      results = []
      images_with_non_person = []

      # Process with multiprocessing
      with Pool(processes=num_processes, initializer=init_worker) as pool:
          for result in tqdm(pool.imap(process_image_pair, args), total=len(args), desc="Processing"):
              if result is None:
                  continue

              results.append((
                  result['name'],
                  result['boxes_real'],
                  result['classes_real'],
                  result['boxes_fake'],
                  result['classes_fake']
              ))

              if result['non_person'] is not None:
                  images_with_non_person.append(result['non_person'])

      # Calculate recall
      recalls = []
      per_class_recalls = {}

      for name, boxes_real, classes_real, boxes_fake, classes_fake in results:
          kept = 0

          for r_box, r_cls in zip(boxes_real, classes_real):
              cls_id = int(r_cls)

              if cls_id not in per_class_recalls:
                  per_class_recalls[cls_id] = []

              matched = False
              for f_box, f_cls in zip(boxes_fake, classes_fake):
                  if r_cls == f_cls and iou(r_box, f_box) > 0.5:
                      kept += 1
                      per_class_recalls[cls_id].append(1)
                      matched = True
                      break

              if not matched:
                  per_class_recalls[cls_id].append(0)

          total = len(boxes_real)
          if total > 0:
              recalls.append(kept / total)

      print(f"\n{'='*60}")
      print(f"RECALL METRICS")
      print(f"{'='*60}")
      if recalls:
          print(f"Overall Accessory Recall: {np.mean(recalls):.4f}")
          print(f"Std Dev: {np.std(recalls):.4f}")
          print(f"Images processed: {len(recalls)}")
          with open('output.txt', 'a') as file:
            file.write(category)
            file.write(": ")
            file.write(np.mean(recalls))
            file.write{"\n"}
      else:
          print("No accessories detected in any images")

# OOT

## paired

In [ ]:
import numpy as np
from ultralytics import YOLO
import cv2
import os
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

# Parameters
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.45
IMG_SIZE = 640

# Global variables for worker processes
_model = None
_person_class_id = None


def prepare_images(model, category):
  files = []
  path_real = f"/content/drive/MyDrive/DressCode/test/{category}"
  path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/{model}/dresscode/unpaired/{category}/"

  for i, name in enumerate(os.listdir(path_real)):
    if "_1" in name:
      continue

    real_file = os.path.join(path_real, name)
    fake_file = os.path.join(path_fake, name)

    if not os.path.isfile(fake_file):
      continue
    else:
      files.append(name)
  return files


def init_worker():
    """Initialize model once per worker process"""
    global _model, _person_class_id
    _model = YOLO("yolo11l-seg.pt")
    _person_class_id = next(
        cls_id for cls_id, cls_name in _model.names.items()
        if cls_name.lower() == "person"
    )

def process_image_pair(args):
    """Process one image pair"""
    name, path_real, path_fake = args

    real_path = os.path.join(path_real, name)
    fake_path = os.path.join(path_fake, name)

    img_real = cv2.imread(real_path)
    img_fake = cv2.imread(fake_path)

    if img_real is None or img_fake is None or img_real.size == 0 or img_fake.size == 0:
        return None

    # Run detection
    det_real = _model.predict(
        img_real,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cpu"
    )

    det_fake = _model.predict(
        img_fake,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device="cpu"
    )

    # Extract detections
    boxes_real = det_real[0].boxes.xyxy.cpu().numpy()
    classes_real = det_real[0].boxes.cls.cpu().numpy()
    confs_real = det_real[0].boxes.conf.cpu().numpy()

    boxes_fake = det_fake[0].boxes.xyxy.cpu().numpy()
    classes_fake = det_fake[0].boxes.cls.cpu().numpy()
    confs_fake = det_fake[0].boxes.conf.cpu().numpy()

    # Check for non-person detections
    non_person_real = [(int(cls), float(conf)) for cls, conf in zip(classes_real, confs_real)
                       if int(cls) != _person_class_id]
    non_person_fake = [(int(cls), float(conf)) for cls, conf in zip(classes_fake, confs_fake)
                       if int(cls) != _person_class_id]

    non_person_info = None
    if non_person_real or non_person_fake:
        detected_classes = {}
        for cls, conf in non_person_real:
            cls_name = _model.names[int(cls)]
            if cls_name not in detected_classes:
                detected_classes[cls_name] = {'real': [], 'fake': []}
            detected_classes[cls_name]['real'].append(f"{conf:.2f}")

        for cls, conf in non_person_fake:
            cls_name = _model.names[int(cls)]
            if cls_name not in detected_classes:
                detected_classes[cls_name] = {'real': [], 'fake': []}
            detected_classes[cls_name]['fake'].append(f"{conf:.2f}")

        non_person_info = {
            'name': name,
            'classes': detected_classes,
            'real_count': len(non_person_real),
            'fake_count': len(non_person_fake)
        }

    return {
        'name': name,
        'boxes_real': boxes_real,
        'classes_real': classes_real,
        'boxes_fake': boxes_fake,
        'classes_fake': classes_fake,
        'non_person': non_person_info
    }

def iou(box1, box2):
    """Calculate Intersection over Union"""
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    box1Area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2Area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    return interArea / (box1Area + box2Area - interArea + 1e-6)

if __name__ == '__main__':
    model = "OOT"
    categories = ["dresses"]
    for category in categories:
      path_real = f"/content/drive/MyDrive/DressCode/test/{category}"
      path_fake = f"/content/drive/MyDrive/DressCode/postvton_experiment/{model}/dresscode/unpaired/{category}/"

      files = prepare_images(model, category)

      # Prepare arguments
      args = [(name, path_real, path_fake) for name in files]

      num_processes = cpu_count()
      print(f"Processing {len(files)} images using {num_processes} CPU cores...")

      results = []
      images_with_non_person = []

      # Process with multiprocessing
      with Pool(processes=num_processes, initializer=init_worker) as pool:
          for result in tqdm(pool.imap(process_image_pair, args), total=len(args), desc="Processing"):
              if result is None:
                  continue

              results.append((
                  result['name'],
                  result['boxes_real'],
                  result['classes_real'],
                  result['boxes_fake'],
                  result['classes_fake']
              ))

              if result['non_person'] is not None:
                  images_with_non_person.append(result['non_person'])

      # Calculate recall
      recalls = []
      per_class_recalls = {}

      for name, boxes_real, classes_real, boxes_fake, classes_fake in results:
          kept = 0

          for r_box, r_cls in zip(boxes_real, classes_real):
              cls_id = int(r_cls)

              if cls_id not in per_class_recalls:
                  per_class_recalls[cls_id] = []

              matched = False
              for f_box, f_cls in zip(boxes_fake, classes_fake):
                  if r_cls == f_cls and iou(r_box, f_box) > 0.5:
                      kept += 1
                      per_class_recalls[cls_id].append(1)
                      matched = True
                      break

              if not matched:
                  per_class_recalls[cls_id].append(0)

          total = len(boxes_real)
          if total > 0:
              recalls.append(kept / total)

      print(f"\n{'='*60}")
      print(f"RECALL METRICS")
      print(f"{'='*60}")
      if recalls:

          print(f"Overall Accessory Recall: {np.mean(recalls):.4f}")
          print(f"Std Dev: {np.std(recalls):.4f}")
          print(f"Images processed: {len(recalls)}")
          with open('output.txt', 'a') as file:
            file.write(category)
            file.write(": ")
            file.write(f"{np.mean(recalls)}")
      else:
          print("No accessories detected in any images")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Processing 1798 images using 2 CPU cores...


Processing:   0%|          | 0/1798 [00:00<?, ?it/s]

Processing:  81%|████████▏ | 1461/1798 [59:30<16:55,  3.01s/it]

In [ ]:
import shutil

shutil.copy("/content/output.txt", "/content/drive/MyDrive/recall.txt")